# 06 Build Comparative Evidence Table

Purpose: integrate the reviewed data-centre and comparator evidence into one synthesis-ready evidence table.

This notebook starts from reviewed evidence only. It does not re-run extraction or deep search. It treats reviewer-filled `verified_*` fields as the authority and keeps job-creation evidence separate from scale/context evidence.


In [ ]:
from pathlib import Path
import re
from datetime import date
import pandas as pd

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 160)

RUN_DATE = date.today().isoformat()


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    marker = Path('Research_Workflow/02_Source_Extraction/data/coding_outputs')
    for p in [start, *start.parents]:
        if (p / marker).exists() and (p / 'Research_Workflow/02_Source_Extraction/notebooks').exists():
            return p
        nested = p / 'DC_job_potential'
        if (nested / marker).exists() and (nested / 'Research_Workflow/02_Source_Extraction/notebooks').exists():
            return nested
    raise FileNotFoundError('Could not locate DC_job_potential project root from current working directory.')

PROJECT_ROOT = find_project_root()
CODED_DIR = PROJECT_ROOT / 'Research_Workflow' / '02_Source_Extraction' / 'data' / 'coding_outputs'
DC_DIR = CODED_DIR / 'data_centre_review' / 'metric_extraction'
CMP_DIR = CODED_DIR / 'comparator_review' / 'metric_extraction'
SYNTHESIS_DIR = PROJECT_ROOT / 'Research_Workflow' / '03_Synthesis' / 'comparative_evidence'
SYNTHESIS_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Synthesis output dir:', SYNTHESIS_DIR)


In [ ]:
paths = {
    'data_centre_synthesis': DC_DIR / 'data_centre_reviewed_synthesis_metrics.csv',
    'data_centre_primary': DC_DIR / 'data_centre_reviewed_citable_metrics.csv',
    'comparator_synthesis': CMP_DIR / 'comparator_reviewed_synthesis_metrics.csv',
    'comparator_primary': CMP_DIR / 'comparator_reviewed_core_job_metrics.csv',
    'data_centre_source_status': CODED_DIR / 'source_recovery_status.csv',
}

missing = [name for name, path in paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required reviewed input files: ' + ', '.join(missing))

print('Input files:')
for name, path in paths.items():
    print(f'- {name}: {path}')


In [ ]:
def read_csv_text(path):
    return pd.read_csv(path, dtype=str, keep_default_na=False)

# Base tables: synthesis files contain all non-rejected/non-duplicate reviewed rows.
dc_synthesis = read_csv_text(paths['data_centre_synthesis'])
cmp_synthesis = read_csv_text(paths['comparator_synthesis'])

# Primary flags: citable/core files are subsets of synthesis files.
dc_primary_ids = set(read_csv_text(paths['data_centre_primary'])['review_id'])
cmp_primary_ids = set(read_csv_text(paths['comparator_primary'])['review_id'])

source_status = read_csv_text(paths['data_centre_source_status'])
dc_url_map = dict(zip(source_status['source_id'], source_status.get('url', '')))

print('Data-centre synthesis rows:', len(dc_synthesis))
print('Data-centre primary/citable rows:', len(dc_primary_ids))
print('Comparator synthesis rows:', len(cmp_synthesis))
print('Comparator primary/core rows:', len(cmp_primary_ids))


In [ ]:
STANDARD_COLUMNS = [
    'evidence_id', 'domain', 'asset_class', 'source_id', 'source_group', 'source_title',
    'review_id', 'review_decision', 'primary_reviewed_flag', 'synthesis_use', 'evidence_role',
    'metric_original', 'metric_family', 'phase', 'verified_value', 'verified_metric_definition',
    'direct_vs_total', 'projected_vs_realised', 'geography', 'currency_price_year',
    'page_or_location', 'source_url', 'source_raw_path', 'text_file', 'notes', 'context_quote',
    'comparison_relevance', 'requires_source_spotcheck', 'manual_split_needed'
]


def coalesce(row, names, default=''):
    for name in names:
        if name in row and str(row[name]).strip():
            return str(row[name]).strip()
    return default


def clean_text(value):
    if pd.isna(value):
        return ''
    return re.sub(r'\s+', ' ', str(value)).strip()


def infer_metric_family(metric, verified_definition, verified_value, notes):
    text = ' '.join([metric, verified_definition, verified_value, notes]).lower()
    if re.search(r'jobs?\s*per\s*(usd|aud|\$|1 million|million)', text) or 'per $1' in text:
        return 'jobs_per_investment'
    if 'job-years/gwh' in text or 'job-years per gwh' in text or 'job years/gwh' in text:
        return 'job_years_per_gwh'
    if 'job-years per mw' in text or 'job-years/mw' in text or 'job years per mw' in text:
        return 'job_years_per_mw'
    if metric == 'jobs_per_mw' or re.search(r'jobs?\s*/\s*mw|jobs?\s+per\s+mw|fte\s*/\s*mw|fte\s+per\s+mw', text):
        return 'jobs_per_mw'
    if metric == 'construction_jobs_peak' or 'peak construction' in text:
        return 'construction_jobs_headcount'
    if metric == 'operational_jobs_fte' or 'operational jobs' in text or 'permanent workers' in text or 'o&m jobs' in text:
        return 'operational_jobs_fte'
    if metric == 'construction_job_years' or 'person-years' in text or 'job-years' in text:
        return 'construction_or_lifecycle_job_years'
    if metric == 'causal_percent_effect' or 'causal' in text or '% change' in text or 'percent' in text:
        return 'causal_local_economy_effect'
    if metric in ['capex', 'capex_or_investment'] or 'capex' in text or 'investment' in text or 'spending' in text or 'pipeline' in text:
        return 'investment_scale'
    if metric in ['capacity_mw', 'capacity_gwh'] or 'mw' in text or 'gwh' in text or 'capacity' in text:
        return 'capacity_scale'
    if 'duration' in metric or 'months' in text or 'years' in text:
        return 'construction_duration'
    return 'other_context'


def infer_phase(metric, phase_guess, verified_definition, notes):
    phase_guess = clean_text(phase_guess).lower()
    text = ' '.join([metric, phase_guess, verified_definition, notes]).lower()
    if phase_guess and phase_guess != 'unspecified':
        if 'construction' in phase_guess:
            return 'construction'
        if 'operation' in phase_guess:
            return 'operation'
        if 'lifecycle' in phase_guess or 'economy' in phase_guess:
            return 'lifecycle_or_economy'
    if 'construction' in text or 'installation' in text or 'build' in text or 'peak' in text:
        return 'construction'
    if 'operation' in text or 'o&m' in text or 'permanent' in text or 'fte' in text:
        return 'operation'
    if 'lifecycle' in text or 'economy' in text or 'causal' in text or 'county' in text:
        return 'lifecycle_or_economy'
    return 'unspecified'


def evidence_role(row, primary_flag):
    decision = clean_text(row.get('ra_decision')).lower()
    family = clean_text(row.get('metric_family'))
    if decision in ['accept', 'accept_scale'] and primary_flag:
        return 'primary_citable_after_spotcheck'
    if decision == 'partial_accept':
        return 'supporting_with_caveats'
    if decision == 'context_only':
        return 'context_only'
    if decision == 'flag_for_verification':
        return 'verify_before_use'
    if primary_flag:
        return 'primary_reviewed'
    if family in ['investment_scale', 'capacity_scale', 'construction_duration']:
        return 'scale_or_context'
    return 'reviewed_synthesis'


def comparison_relevance(metric_family, role):
    if role == 'context_only':
        return 'narrative_context_only'
    if metric_family in ['jobs_per_investment', 'jobs_per_mw', 'job_years_per_mw', 'job_years_per_gwh']:
        return 'normalisable_job_intensity'
    if metric_family in ['construction_jobs_headcount', 'operational_jobs_fte', 'construction_or_lifecycle_job_years']:
        return 'absolute_job_or_job_year_benchmark'
    if metric_family == 'causal_local_economy_effect':
        return 'causal_local_economy_effect'
    if metric_family in ['investment_scale', 'capacity_scale', 'construction_duration']:
        return 'scale_context_not_job_creation'
    return 'supporting_context'


def needs_manual_split(value):
    # Many reviewed rows intentionally contain multiple category values in one cell.
    text = clean_text(value)
    nums = re.findall(r'(?<!\w)[<>~]?[0-9]+(?:\.[0-9]+)?', text.replace(',', ''))
    return len(nums) > 1


def standardise(df, domain, primary_ids):
    records = []
    for _, row in df.iterrows():
        review_id = coalesce(row, ['review_id'])
        primary_flag = review_id in primary_ids
        metric = coalesce(row, ['metric'])
        verified_definition = clean_text(coalesce(row, ['verified_metric_definition']))
        verified_value = clean_text(coalesce(row, ['verified_value', 'raw_value']))
        notes = clean_text(coalesce(row, ['notes']))
        metric_family = infer_metric_family(metric, verified_definition, verified_value, notes)
        phase = infer_phase(metric, coalesce(row, ['phase_guess']), verified_definition, notes)
        role = evidence_role({**row.to_dict(), 'metric_family': metric_family}, primary_flag)
        source_url = coalesce(row, ['source_url'])
        if domain == 'data_centre' and not source_url:
            source_url = dc_url_map.get(coalesce(row, ['source_id']), '')
        records.append({
            'domain': domain,
            'asset_class': coalesce(row, ['asset_type'], 'data_centre' if domain == 'data_centre' else 'comparator'),
            'source_id': coalesce(row, ['source_id']),
            'source_group': coalesce(row, ['source_group']),
            'source_title': coalesce(row, ['title']),
            'review_id': review_id,
            'review_decision': coalesce(row, ['ra_decision']),
            'primary_reviewed_flag': primary_flag,
            'synthesis_use': coalesce(row, ['synthesis_use']),
            'evidence_role': role,
            'metric_original': metric,
            'metric_family': metric_family,
            'phase': phase,
            'verified_value': verified_value,
            'verified_metric_definition': verified_definition,
            'direct_vs_total': coalesce(row, ['direct_vs_total']),
            'projected_vs_realised': coalesce(row, ['projected_vs_realised']),
            'geography': coalesce(row, ['geography']),
            'currency_price_year': coalesce(row, ['currency_price_year']),
            'page_or_location': coalesce(row, ['page_or_location']),
            'source_url': source_url,
            'source_raw_path': coalesce(row, ['source_raw_path']),
            'text_file': coalesce(row, ['text_file']),
            'notes': notes,
            'context_quote': clean_text(coalesce(row, ['context_quote'])),
            'comparison_relevance': comparison_relevance(metric_family, role),
            'requires_source_spotcheck': bool(re.search(r'cross-check|spot-check|context_quote|pdf', notes.lower())) or True,
            'manual_split_needed': needs_manual_split(verified_value),
        })
    out = pd.DataFrame(records)
    out.insert(0, 'evidence_id', [f'ev_{i:04d}' for i in range(1, len(out) + 1)])
    return out[STANDARD_COLUMNS]

master = pd.concat([
    standardise(dc_synthesis, 'data_centre', dc_primary_ids),
    standardise(cmp_synthesis, 'comparator', cmp_primary_ids),
], ignore_index=True)

print('Master evidence rows:', len(master))
display(pd.crosstab(master['domain'], master['evidence_role']))
display(pd.crosstab(master['domain'], master['metric_family']))


In [ ]:
primary_table = master[master['primary_reviewed_flag'] == True].copy()
job_intensity_candidates = master[
    master['comparison_relevance'].isin([
        'normalisable_job_intensity',
        'absolute_job_or_job_year_benchmark',
        'causal_local_economy_effect',
    ])
].copy()
scale_context = master[master['comparison_relevance'].eq('scale_context_not_job_creation')].copy()
context_only = master[master['comparison_relevance'].eq('narrative_context_only')].copy()

source_summary = (
    master.groupby(['domain', 'asset_class', 'source_id', 'source_title'], dropna=False)
    .agg(
        evidence_rows=('evidence_id', 'count'),
        primary_rows=('primary_reviewed_flag', 'sum'),
        metric_families=('metric_family', lambda s: ', '.join(sorted(set(s)))),
        evidence_roles=('evidence_role', lambda s: ', '.join(sorted(set(s)))),
        review_decisions=('review_decision', lambda s: ', '.join(sorted(set(s)))),
    )
    .reset_index()
    .sort_values(['domain', 'primary_rows', 'evidence_rows'], ascending=[True, False, False])
)

print('Primary table rows:', len(primary_table))
print('Job/intensity candidate rows:', len(job_intensity_candidates))
print('Scale/context rows:', len(scale_context))
print('Context-only rows:', len(context_only))
display(source_summary.head(20))


In [ ]:
gap_log = pd.DataFrame([
    {
        'gap_id': 'gap_001',
        'gap_type': 'source_verification',
        'priority': 'high',
        'issue': 'Final source/PDF spot-checks are still needed before citation, especially rows reviewed from context_quote snippets.',
        'recommended_action': 'Spot-check page_or_location/source_raw_path for every primary row used in manuscript tables.',
    },
    {
        'gap_id': 'gap_002',
        'gap_type': 'comparability',
        'priority': 'high',
        'issue': 'Direct jobs, total supported jobs, induced jobs, job-years, and causal percentage effects are mixed across sources.',
        'recommended_action': 'Keep metric_family and direct_vs_total visible in tables. Do not average unlike units.',
    },
    {
        'gap_id': 'gap_003',
        'gap_type': 'currency_and_price_year',
        'priority': 'medium',
        'issue': 'Investment figures use different currencies, years, and sometimes unspecified price bases.',
        'recommended_action': 'Normalise to AUD 2026 only in a separate calculation table after source spot-checking.',
    },
    {
        'gap_id': 'gap_004',
        'gap_type': 'asset_comparator_coverage',
        'priority': 'medium',
        'issue': 'Comparator evidence is stronger for energy, infrastructure, and battery manufacturing than for high-rise or commercial building assets of similar investment scale.',
        'recommended_action': 'Consider a targeted gap search for high-rise/commercial construction jobs per capex, peak workforce, or job-years.',
    },
    {
        'gap_id': 'gap_005',
        'gap_type': 'australian_localisation',
        'priority': 'medium',
        'issue': 'Australian data-centre capacity and skills context exists, but direct Australian job multipliers are still limited.',
        'recommended_action': 'Use NSW capacity/skills rows for Australian scaling context and label US-derived job intensity assumptions explicitly.',
    },
])

display(gap_log)


In [ ]:
out_master = SYNTHESIS_DIR / 'comparative_evidence_master_table.csv'
out_primary = SYNTHESIS_DIR / 'comparative_evidence_primary_table.csv'
out_jobs = SYNTHESIS_DIR / 'jobs_intensity_comparison_for_figures.csv'
out_scale = SYNTHESIS_DIR / 'scale_context_table.csv'
out_source_summary = SYNTHESIS_DIR / 'source_level_synthesis_summary.csv'
out_gap = SYNTHESIS_DIR / 'evidence_gap_log.csv'
out_xlsx = SYNTHESIS_DIR / 'comparative_evidence_tables.xlsx'

master.to_csv(out_master, index=False)
primary_table.to_csv(out_primary, index=False)
job_intensity_candidates.to_csv(out_jobs, index=False)
scale_context.to_csv(out_scale, index=False)
source_summary.to_csv(out_source_summary, index=False)
gap_log.to_csv(out_gap, index=False)

readme = pd.DataFrame({
    'item': [
        'created', 'basis', 'master_rows', 'primary_rows', 'job_intensity_candidate_rows',
        'main_warning', 'next_step'
    ],
    'value': [
        RUN_DATE,
        'Reviewed data-centre and reviewed comparator extraction tables only',
        len(master), len(primary_table), len(job_intensity_candidates),
        'Do not average unlike units. Keep direct_vs_total, projected_vs_realised, and metric_family visible.',
        'Use primary table as the manuscript evidence starting point; keep caveats visible and spot-check original sources.'
    ]
})

with pd.ExcelWriter(out_xlsx, engine='openpyxl') as writer:
    readme.to_excel(writer, sheet_name='README', index=False)
    master.to_excel(writer, sheet_name='master_evidence', index=False)
    primary_table.to_excel(writer, sheet_name='primary_table', index=False)
    job_intensity_candidates.to_excel(writer, sheet_name='job_figure_candidates', index=False)
    scale_context.to_excel(writer, sheet_name='scale_context', index=False)
    source_summary.to_excel(writer, sheet_name='source_summary', index=False)
    gap_log.to_excel(writer, sheet_name='gap_log', index=False)

    # Light formatting for readability.
    workbook = writer.book
    for ws in workbook.worksheets:
        ws.freeze_panes = 'A2'
        ws.auto_filter.ref = ws.dimensions
        for col in ws.columns:
            header = str(col[0].value or '')
            width = min(max(len(header) + 2, 12), 42)
            for cell in col[1:80]:
                width = min(max(width, min(len(str(cell.value or '')) + 2, 42)), 42)
            ws.column_dimensions[col[0].column_letter].width = width
            for cell in col:
                cell.alignment = cell.alignment.copy(wrap_text=True, vertical='top')

print('Wrote:')
for path in [out_master, out_primary, out_jobs, out_scale, out_source_summary, out_gap, out_xlsx]:
    print('-', path)


In [ ]:
summary_md = SYNTHESIS_DIR / 'COMPARATIVE_SYNTHESIS_NOTES.md'

role_counts = master['evidence_role'].value_counts().to_dict()
family_counts = master['metric_family'].value_counts().to_dict()
domain_counts = master['domain'].value_counts().to_dict()

notes = f'''# Comparative Evidence Synthesis Notes

Date: {RUN_DATE}

## Inputs

This synthesis uses reviewed evidence only:

- Data-centre reviewed synthesis rows: {len(dc_synthesis):,}
- Data-centre primary/citable rows: {len(dc_primary_ids):,}
- Comparator reviewed synthesis rows: {len(cmp_synthesis):,}
- Comparator primary/core rows: {len(cmp_primary_ids):,}

## Outputs

- `comparative_evidence_master_table.csv`: all reviewed synthesis rows, standardised into one schema.
- `comparative_evidence_primary_table.csv`: cleanest rows for manuscript tables after source spot-checking.
- `jobs_intensity_comparison_for_figures.csv`: rows that may support a figure or comparison of job intensity, job headcount, job-years, or causal effects.
- `scale_context_table.csv`: investment, capacity, and duration rows. These should contextualise scale, not imply job creation by themselves.
- `source_level_synthesis_summary.csv`: source-level counts and metric families.
- `evidence_gap_log.csv`: remaining methodological/source gaps.
- `comparative_evidence_tables.xlsx`: workbook version of the tables.

## Current State

The master table contains {len(master):,} rows: {domain_counts.get('data_centre', 0):,} data-centre rows and {domain_counts.get('comparator', 0):,} comparator rows.

The primary table contains {len(primary_table):,} rows. These are the reviewed data-centre citable rows plus the comparator core rows. Some core comparator rows remain caveated, so row-level `evidence_role` and `notes` must stay visible.

## Core Interpretation

The reviewed evidence supports a cautious thesis: data centres can create substantial temporary construction work, especially during peak build phases, but long-term direct operational employment is comparatively modest relative to capital investment and capacity. Economic-impact studies often report total supported jobs or multiplier effects, which should not be compared directly with direct FTE or peak on-site construction headcount.

The comparator evidence helps put this in perspective. Renewable energy and energy-efficiency studies provide jobs-per-investment and job-year benchmarks, while Infrastructure Australia and battery manufacturing sources provide investment-scale context. These are useful comparators, but the table keeps job-intensity evidence separate from scale-only evidence to avoid overclaiming.

## Immediate Use Rules

1. Use `comparative_evidence_primary_table.csv` as the first manuscript evidence table, keeping `evidence_role` and `notes` visible.
2. Use `jobs_intensity_comparison_for_figures.csv` only after manually splitting rows that contain multiple values in one cell.
3. Use `scale_context_table.csv` for investment/capacity perspective only.
4. Keep `direct_vs_total`, `projected_vs_realised`, and `metric_family` visible in any paper table.
5. Do not average data-centre and comparator metrics unless they share the same unit and scope.

## Remaining Gaps

The main remaining gap is comparator coverage for high-rise or commercial building assets of similar investment scale. The current comparator set is stronger for energy, infrastructure, and advanced manufacturing than for high-rise construction. A targeted gap search may still be worthwhile if the paper needs a direct building-sector comparator.
'''

summary_md.write_text(notes, encoding='utf-8')
print(summary_md)
print(notes)


In [ ]:
# Compact QA checks.
assert len(master) == len(dc_synthesis) + len(cmp_synthesis), 'Master row count mismatch.'
assert master['review_id'].is_unique, 'review_id values should be unique across data-centre and comparator evidence.'
assert not master['source_title'].eq('').any(), 'Some rows are missing source_title.'
assert (primary_table['primary_reviewed_flag'] == True).all(), 'Primary table contains non-primary rows.'

print('QA passed.')
print('Rows by domain:')
display(master['domain'].value_counts().to_frame('rows'))
print('Rows by comparison relevance:')
display(master['comparison_relevance'].value_counts().to_frame('rows'))
